# MT-SOHO Phase 1A — CIFAR-100 train-only
This notebook tests fixed-WTA analytic moment transport. It never opens the CIFAR-100 test split.

In [ ]:
# === Edit this cell only ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
WORK_DIR = '/content/SOHO-CL'
LOCAL_CACHE_DIR = '/content/mt_soho_cifar100_train_cache'
DRIVE_TRAIN_CACHE = '/content/drive/MyDrive/T-SOHO/tsoho_cifar100_cache'
SAVE_PROGRESS_TO_DRIVE = False
LOCAL_OUTPUT_DIR = '/content/mt_soho_phase1a_outputs'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/T-SOHO/mt_soho_phase1a_outputs'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_CONFIG_SHA256 = '395cccdca828ae462c58bab5371b08ea959da0df37f0b3e8b5f5fd0e0200bfbc'
EXPECTED_RUNNER_SHA256 = '353b59935a8ece2b5694237f662c1e12ea8b64352522c0f093fb28a233e2c8ab'

In [ ]:
# Fresh repository and environment.
import os, sys, json, shutil, subprocess, hashlib, zipfile, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sha('configs/mt_soho_phase1a_cifar100_train_only.json')==EXPECTED_CONFIG_SHA256
assert sha('tools/mt_soho_phase1.py')==EXPECTED_RUNNER_SHA256
print('repo commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('LOCKED CONFIG/RUNNER HASH: PASS')

In [ ]:
# Synthetic mathematical, state, and runner gates.
completed=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_mt_soho.py','tests/test_mt_soho_phase1.py'])
assert completed.returncode==0,'Correctness gate failed; return the full pytest output.'
print('MT-SOHO CORRECTNESS GATE: PASS')

In [ ]:
# Restore a train-only cache; if unavailable, extract train features once with live task progress.
import torch
cache=Path(LOCAL_CACHE_DIR); drive_cache=Path(DRIVE_TRAIN_CACHE)
if cache.exists(): shutil.rmtree(cache)
cache.mkdir(parents=True)
if (drive_cache/'train.pt').is_file() and (drive_cache/'metadata.json').is_file():
    shutil.copy2(drive_cache/'train.pt',cache/'train.pt'); shutil.copy2(drive_cache/'metadata.json',cache/'metadata.json')
    print('Restored training cache from Drive.')
else:
    import kagglehub
    from huggingface_hub import hf_hub_download
    downloaded=Path(kagglehub.dataset_download('zaphat206/cifar-100'))
    processed=Path('/content/processed_datasets'); processed.mkdir(exist_ok=True)
    link=processed/'cifar-100'
    if link.exists() or link.is_symlink(): link.unlink() if link.is_symlink() else shutil.rmtree(link)
    os.symlink(downloaded,link,target_is_directory=True)
    checkpoint=hf_hub_download('timm/vit_base_patch16_224.augreg2_in21k_ft_in1k','model.safetensors')
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',str(processed),'--backbone-checkpoint',checkpoint,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',str(cache),'--output-dir','/content/mt_soho_cache_extract','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('No reusable cache: extracting TRAIN only; one progress line per task.',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and (cache/'metadata.json').is_file()
assert not (cache/'test.pt').exists(),'FAIL: test.pt is visible'
train=torch.load(cache/'train.pt',weights_only=True,map_location='cpu')
assert tuple(train['features'].shape)==(50000,768) and torch.isfinite(train['features']).all()
print('TRAIN-ONLY CACHE PASS:',tuple(train['features'].shape)); del train

In [ ]:
# Locked nested train-only study. Every completed unit is resumable from OUTPUT_DIR.
OUTPUT_DIR = DRIVE_OUTPUT_DIR if SAVE_PROGRESS_TO_DRIVE else LOCAL_OUTPUT_DIR
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u','tools/mt_soho_phase1.py','--config','configs/mt_soho_phase1a_cifar100_train_only.json','--feature-cache-dir',LOCAL_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda']
print('Starting Phase 1A: 4 anchor settings, 16 MT candidates, then four outer controls.',flush=True)
completed=subprocess.run(command)
assert completed.returncode==0,'Phase 1A failed; return the complete traceback without editing the grid.'
result=json.loads(Path(OUTPUT_DIR,'phase1_results.json').read_text())
print('PHASE 1A:',result['status']); print(json.dumps(result['gates'],indent=2))

In [ ]:
# Compact result table and evidence download.
import pandas as pd
result=json.loads(Path(OUTPUT_DIR,'phase1_results.json').read_text())
rows=[]
for method,values in result['outer_validation'].items():
    rows.append({'method':method,'mean_outer_AIA':sum(v['average_incremental_accuracy'] for v in values)/len(values),'mean_final_accuracy':sum(v['final_accuracy'] for v in values)/len(values),'state_MiB':sum(v['persistent_state_bytes'] for v in values)/len(values)/2**20})
display(pd.DataFrame(rows).sort_values('mean_outer_AIA',ascending=False))
archive=Path('/content/mt_soho_phase1a_train_only.zip')
if archive.exists(): archive.unlink()
shutil.make_archive(str(archive.with_suffix('')),'zip',OUTPUT_DIR)
print('artifact:',archive,'bytes=',archive.stat().st_size,'sha256=',hashlib.sha256(archive.read_bytes()).hexdigest())
from google.colab import files
files.download(str(archive))